# API paper/sim market-data adapter v1

目标：补齐 `src/api` 的 data adapter contract。v1 只支持 paper/sim，从 canonical/dataframe 读取 as-of quote，不连接真实 broker，也不触发真实下单。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.api import (
    PaperBrokerAdapter,
    PaperMarketDataAdapter,
    build_target_position_intent_from_dashboard,
    execute_paper_intent,
)

EXPERIMENT_DIR = ROOT / "research" / "api_market_data_adapter_v1"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

price_frame = pd.DataFrame(
    [
        {"date": "2026-01-02", "instrument": "SOXL", "open": 10.0, "close": 10.5, "volume": 1000},
        {"date": "2026-01-05", "instrument": "SOXL", "open": 11.0, "close": 11.5, "volume": 1200},
    ]
)
canonical_path = EXPERIMENT_DIR / "paper_market_data_frame.csv"
price_frame.to_csv(canonical_path, index=False)
price_frame

,date,instrument,open,close,volume
0,2026-01-02,SOXL,10.0,10.5,1000
1,2026-01-05,SOXL,11.0,11.5,1200


In [2]:
market_data = PaperMarketDataAdapter.from_csv(canonical_path, as_of="2026-01-05")
quote = market_data.latest_quote("SOXL")

assert market_data.adapter_name == "paper_market_data"
assert quote.schema_version == "api.market_quote.v1"
assert quote.instrument == "SOXL"
assert quote.date == "2026-01-05"
assert quote.price == 11.5
assert quote.data_mode == "paper"
quote.to_dict()

{'schema_version': 'api.market_quote.v1',
 'instrument': 'SOXL',
 'date': '2026-01-05',
 'price': 11.5,
 'field': 'close',
 'data_mode': 'paper',
 'source_path': '/Users/rziiiii/dev/workspace/research/api_market_data_adapter_v1/paper_market_data_frame.csv'}

In [3]:
future_frame = pd.concat(
    [price_frame, pd.DataFrame([{"date": "2026-01-06", "instrument": "SOXL", "open": 12.0, "close": 12.5, "volume": 1300}])],
    ignore_index=True,
)
future_guard_error = ""
try:
    PaperMarketDataAdapter.from_frame(future_frame, as_of="2026-01-05")
except ValueError as exc:
    future_guard_error = str(exc)

assert "future market data row" in future_guard_error
future_guard_error

'future market data row after as_of=2026-01-05: first=2026-01-06'

In [4]:
dashboard_payload = {
    "schema_version": "dashboard.model_matrix.v1",
    "artifact_id": "api_market_data_adapter_v1_unit_dashboard",
    "top_models": [{"model": "ridge_mse", "avg_position": 0.5, "stability_score": 10.0}],
}
intent = build_target_position_intent_from_dashboard(dashboard_payload, instrument="SOXL")
broker = PaperBrokerAdapter(cash=1000.0, shares=0)
result = execute_paper_intent(intent, broker=broker, price=quote.price)

assert intent.mode == "paper"
assert intent.target_position == 0.5
assert result.status == "paper_executed"
assert result.snapshot.price == quote.price
assert result.snapshot.shares == 43
result.to_dict()

{'schema_version': 'api.paper_execution_result.v1',
 'status': 'paper_executed',
 'broker_name': 'paper',
 'intent': {'instrument': 'SOXL',
  'target_position': 0.5,
  'source_model': 'ridge_mse',
  'source_artifact': 'api_market_data_adapter_v1_unit_dashboard',
  'mode': 'paper',
  'schema_version': 'api.execution_intent.v1',
  'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
  'metadata': {'execution_gate': 'paper_review',
   'raw_avg_position': 0.5,
   'min_position': 0.0,
   'max_position': 1.0,
   'base_model': '',
   'feature_set': '',
   'stability_score': 10.0,
   'max_drawdown_pct': 0.0,
   'promotion_decision': '',
   'promotion_score': 0.0,
   'promotion_blockers': '',
   'dashboard_schema_version': 'dashboard.model_matrix.v1',
   'run_status': 'unknown',
   'run_trained': False,
   'run_predicted': False,
   'run_backtested': False,
   'run_objective_evaluated': False,
   'run_promotion_evaluated': False,
   'run_analysis_recorded': False,
   'run_manifested': False,


In [5]:
summary = {
    "schema_version": "research.api_market_data_adapter_v1",
    "status": "completed",
    "adapter_name": market_data.adapter_name,
    "quote": quote.to_dict(),
    "future_guard_error": future_guard_error,
    "intent": intent.to_dict(),
    "execution_result": result.to_dict(),
    "live_broker_enabled": False,
}
(EXPERIMENT_DIR / "market_quote.json").write_text(json.dumps(quote.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")
(EXPERIMENT_DIR / "paper_execution_result.json").write_text(json.dumps(result.to_dict(), ensure_ascii=False, indent=2), encoding="utf-8")
(EXPERIMENT_DIR / "analysis_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary

{'schema_version': 'research.api_market_data_adapter_v1',
 'status': 'completed',
 'adapter_name': 'paper_market_data',
 'quote': {'schema_version': 'api.market_quote.v1',
  'instrument': 'SOXL',
  'date': '2026-01-05',
  'price': 11.5,
  'field': 'close',
  'data_mode': 'paper',
  'source_path': '/Users/rziiiii/dev/workspace/research/api_market_data_adapter_v1/paper_market_data_frame.csv'},
 'future_guard_error': 'future market data row after as_of=2026-01-05: first=2026-01-06',
 'intent': {'instrument': 'SOXL',
  'target_position': 0.5,
  'source_model': 'ridge_mse',
  'source_artifact': 'api_market_data_adapter_v1_unit_dashboard',
  'mode': 'paper',
  'schema_version': 'api.execution_intent.v1',
  'guardrail': 'v1 只允许 paper/sim，不连接真实 broker，不触发真实下单。',
  'metadata': {'execution_gate': 'paper_review',
   'raw_avg_position': 0.5,
   'min_position': 0.0,
   'max_position': 1.0,
   'base_model': '',
   'feature_set': '',
   'stability_score': 10.0,
   'max_drawdown_pct': 0.0,
   'promoti

## 中文结论

- 做了什么：为 `src/api` 增加 paper/sim market-data adapter 证据面，从 canonical CSV/dataframe 读取 SOXL as-of quote，并把 quote price 传入 paper intent 执行。
- 核心指标：adapter=`paper_market_data`，quote schema=`api.market_quote.v1`，as_of quote date=`2026-01-05`，price=`11.5`，paper shares=`43`。
- regime 差异：本 notebook 不做 regime 分析；regime/model 差异仍由 `src/analysis` 和 QuantBench ledger/report notebook 负责。
- 是否晋升：否。本轮只补 API data adapter contract，不晋升 live broker，也不新增真实下单能力。
- 风险：paper quote 只来自已发布/传入的 canonical 数据；未来行会被拒绝。后续 production notebook 应通过 manifest/data registry 选择具体 snapshot，而不是硬编码散落价格文件。
- 下一步：把 `PaperMarketDataAdapter.from_csv` 接入 production paper/sim replay，使 replay 同时记录 quote manifest/source path。